In [1]:
## imports
import pandas as pd
import numpy as np
import re
import requests
import yaml


## repeated printouts
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# 1. Example 1: no credentials; no wrapper

Site: National Assessment of Education Progress (NAEP)

Documentation: https://www.nationsreportcard.gov/api_documentation.aspx

Base link: https://www.nationsreportcard.gov/DataService/GetAdhocData.aspx 

## 1.1 Query to pull some data

In [2]:
## using their example query of 2011 writing scores separated by gender
## based on here - https://stackoverflow.com/questions/40836749/pythonic-way-of-writing-a-single-line-long-string
## using the ( ) syntax to formulate a long
## string without linebreaks added
example_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011')


example_naep_query


'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2011'

In [3]:
## use requests to call the api
naep_resp = requests.get(example_naep_query)
naep_resp
print(type(naep_resp))

## get the json contents of the response 
## here, we're assuming valid response
naep_resp_j = naep_resp.json()
naep_resp_j

## with result, turn it into a dataframe
naep_resp_d = pd.DataFrame(naep_resp_j['result'])
naep_resp_d

<Response [200]>

<class 'requests.models.Response'>


{'status': 200,
 'serviceVersion': '6.4.2026.1',
 'dwellTimeMS': '15.6285',
 'avgWebHostCPUTotalLoad': 'N/A',
 'dataHitType': 'FROM_MEMORY',
 'Source': 'B11A',
 'result': [{'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '1',
   'varValueLabel': 'Male',
   'value': 139.099504632971,
   'isStatDisplayable': 1,
   'errorFlag': 0},
  {'year': 2011,
   'sample': 'R3',
   'yearSampleLabel': '2011',
   'Cohort': 2,
   'CohortLabel': 'Grade 8',
   'stattype': 'MN:MN',
   'subject': 'WRI',
   'grade': 8,
   'scale': 'WRIRP',
   'jurisdiction': 'NP',
   'jurisLabel': 'National public',
   'variable': 'GENDER',
   'variableLabel': 'Sex',
   'varValue': '2',
   'varValueLabel': 'Female',
   'value': 158.567104984955,
   'isStatDispl

,year,sample,yearSampleLabel,Cohort,CohortLabel,stattype,subject,grade,scale,jurisdiction,jurisLabel,variable,variableLabel,varValue,varValueLabel,value,isStatDisplayable,errorFlag
0,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,1,Male,139.099505,1,0
1,2011,R3,2011,2,Grade 8,MN:MN,WRI,8,WRIRP,NP,National public,GENDER,Sex,2,Female,158.567105,1,0


## 1.2 What happens if there's an error in our query?

In [4]:
## here's a query that from the documentation we know
## won't work since i modified year to 2025 which doesnt
## exist in the data
wrong_naep_query = (
'https://www.nationsreportcard.gov/'
'Dataservice/GetAdhocData.aspx?'
'type=data&subject=writing&grade=8&'
'subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025')

wrong_naep_query

'https://www.nationsreportcard.gov/Dataservice/GetAdhocData.aspx?type=data&subject=writing&grade=8&subscale=WRIRP&variable=GENDER&jurisdiction=NP&stattype=MN:MN&Year=2025'

In [5]:
## use requests to call the api
naep_wrong_resp = requests.get(wrong_naep_query)
naep_wrong_resp

<Response [400]>

In [6]:
## in the case of this particular api,
## the call returns some response but
## when we try to extract the json containing
## status or results, we get in an error
#naep_wrong_resp.json() # uncomment to see error

### 1.2.2 More all-purpose way of allowing remainder of calls to run: try, except

In [7]:
## putting it in a try; except as general error catching
try:
    results = naep_wrong_resp.json()['result']
except Exception as e:
    print('Failed to get result from API due to error:')
    print(e) # or just: pass

Failed to get result from API due to error:
Invalid control character at: line 1 column 293 (char 292)


### 1.2.3 Can usually also find more targeted way but that varies more across APIs

In [ ]:
## if we wanted do more specific error catching,
## see that the status == 400 actually appears here
## so could write if else along those lines
naep_wrong_resp.text
naep_resp.text

if "System.Exception" in naep_wrong_resp.text:
    print("NAEP results not found")

## Activity 1: writing a function to make multiple, sequential calls

- Say we want to pull the data for grades 4, 8, and 12
- How can we write a function that iterates over a list of those grades and pulls the data for each grade?

**Note**: an ideal function would have arguments for each parameter in the API like subject, subscale, etc. Here we can leave those other parts constant

In [8]:
# your code here

# 2. Example 2: needs credentials; no wrapper

Create an account here: https://www.yelp.com/developers/v3/manage_app

In [9]:
## get the key
API_KEY = "t1A8Xqn1RN9hZYnHmLGmZ2q-30-Mqk7noeOtqNoEbbWxwmWlDWB0zTVLUuCMapIGa97d8pyb9c_K27E-IREpURt-gFUhMQpo692L6_jepFObCBwMQTPhsWwExpp0anYx"

In [10]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Hanover,NH,03755"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()


<Response [200]>

In [12]:
## example business
yelp_genjson['businesses'][0]

## more automatic way of summarizing but things end up in lists
## within columns for things like categories
yelp_gendf = pd.DataFrame(yelp_genjson['businesses'])
yelp_gendf.head()

{'id': '8ybF6YyRldtZmU9jil4xlg',
 'alias': 'mollys-restaurant-and-bar-hanover',
 'name': "Molly's Restaurant & Bar",
 'image_url': 'https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA6z-SnPgZfrs2GQNQ/o.jpg',
 'is_closed': False,
 'url': 'https://www.yelp.com/biz/mollys-restaurant-and-bar-hanover?adjust_creative=eQYW8f2PoP2KTv7QR4Sa2A&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_search&utm_source=eQYW8f2PoP2KTv7QR4Sa2A',
 'review_count': 582,
 'categories': [{'alias': 'tradamerican', 'title': 'American'},
  {'alias': 'burgers', 'title': 'Burgers'},
  {'alias': 'pizza', 'title': 'Pizza'}],
 'rating': 3.9,
 'coordinates': {'latitude': 43.701144, 'longitude': -72.2894249},
 'transactions': ['delivery'],
 'price': '$$',
 'location': {'address1': '43 South Main St',
  'address2': '',
  'address3': '',
  'city': 'Hanover',
  'zip_code': '03755',
  'country': 'US',
  'state': 'NH',
  'display_address': ['43 South Main St', 'Hanover, NH 03755']},
 'phone': '+16036432570',
 'display_phone': '(

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,coordinates,transactions,price,location,phone,display_phone,distance,business_hours,attributes
0,8ybF6YyRldtZmU9jil4xlg,mollys-restaurant-and-bar-hanover,Molly's Restaurant & Bar,https://s3-media0.fl.yelpcdn.com/bphoto/TJLrrA...,False,https://www.yelp.com/biz/mollys-restaurant-and...,582,"[{'alias': 'tradamerican', 'title': 'American'...",3.9,"{'latitude': 43.701144, 'longitude': -72.2894249}",[delivery],$$,"{'address1': '43 South Main St', 'address2': '...",+16036432570,(603) 643-2570,250.830160,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://www.mollysrestaurant.com/...
1,JFE0XffhpP3Bi3DQRvymVg,little-havana-hanover,Little Havana,https://s3-media0.fl.yelpcdn.com/bphoto/941vaR...,False,https://www.yelp.com/biz/little-havana-hanover...,29,"[{'alias': 'cuban', 'title': 'Cuban'}, {'alias...",4.9,"{'latitude': 43.700743, 'longitude': -72.287599}","[delivery, pickup]",NaN,"{'address1': '15 Lebanon St', 'address2': '', ...",+18383831000,(838) 383-1000,102.833229,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://www.canva.com/design/DAG...
2,XVGEEIH5rVB2QzW-qywcJw,base-camp-cafe-hanover,Base Camp Cafe,https://s3-media0.fl.yelpcdn.com/bphoto/tScZeo...,False,https://www.yelp.com/biz/base-camp-cafe-hanove...,264,"[{'alias': 'himalayan', 'title': 'Himalayan/Ne...",4.4,"{'latitude': 43.700626, 'longitude': -72.2887803}",[delivery],$$,"{'address1': '3 Lebanon St', 'address2': 'Ste ...",+16036432007,(603) 643-2007,196.139758,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'http://basecampcafenewhampshire....
3,wyV_NfYn4ZOfp_sHMDPcAw,bistro-at-six-hanover,Bistro at Six,https://s3-media0.fl.yelpcdn.com/bphoto/i4jvss...,False,https://www.yelp.com/biz/bistro-at-six-hanover...,2,"[{'alias': 'lounges', 'title': 'Lounges'}, {'a...",4.0,"{'latitude': 43.7001146, 'longitude': -72.2877...",[],$$,"{'address1': '6 E South St', 'address2': '', '...",+16036430600,(603) 643-0600,198.651788,"[{'open': [{'is_overnight': True, 'start': '00...",{}
4,1Q9gTry0GH2NFA7O398xeA,casa-brava-tapas-hanover,Casa Brava Tapas,https://s3-media0.fl.yelpcdn.com/bphoto/fA3__V...,False,https://www.yelp.com/biz/casa-brava-tapas-hano...,10,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.8,"{'latitude': 43.70019133305323, 'longitude': -...",[],NaN,"{'address1': '6 South St', 'address2': '', 'ad...",+16038509763,(603) 850-9763,202.701925,"[{'open': [{'is_overnight': False, 'start': '1...",{'menu_url': 'https://casabravatapas.com/tapas...


In [51]:
## more data-specific way of summarizing
## we're doing a simple approach and just retaining
## cols that have a simple str structure
## if doing for real, would want to extract things
def clean_yelp_json(one_biz):

    ## restrict to str cols
    d_str = {key:value for key, value in one_biz.items()
             if type(value) == str}
    
    df_str = pd.DataFrame(d_str, index = [d_str['id']])
    return(df_str)

yelp_stronly = [clean_yelp_json(one_b) for one_b in yelp_genjson['businesses']]
yelp_stronly_df = pd.concat(yelp_stronly)

yelp_stronly_df.head(7)


,id,alias,name,image_url,url,phone,display_phone,price
-4fGYNfRa-hiiU1hCVFUEw,-4fGYNfRa-hiiU1hCVFUEw,twinflame-oakland,Twinflame,https://s3-media0.fl.yelpcdn.com/bphoto/76RheI...,https://www.yelp.com/biz/twinflame-oakland?adj...,,,NaN
L7I_bhmMacR1lcVUuWwuKg,L7I_bhmMacR1lcVUuWwuKg,almond-and-oak-oakland-2,Almond & Oak,https://s3-media0.fl.yelpcdn.com/bphoto/8vYz42...,https://www.yelp.com/biz/almond-and-oak-oaklan...,+15102509550,(510) 250-9550,$$
D_yAQ4tHlWp2CJSqFCeW_w,D_yAQ4tHlWp2CJSqFCeW_w,mama-oakland-oakland,MAMA Oakland,https://s3-media0.fl.yelpcdn.com/bphoto/uH7mYb...,https://www.yelp.com/biz/mama-oakland-oakland?...,+15109746372,(510) 974-6372,$$$
5x-d9REDONrMabzr9GBm4A,5x-d9REDONrMabzr9GBm4A,sirene-oakland-2,Sirene,https://s3-media0.fl.yelpcdn.com/bphoto/K91CJr...,https://www.yelp.com/biz/sirene-oakland-2?adju...,+15102008750,(510) 200-8750,NaN
Q7RiDz9I2PpltrOnnIfDyg,Q7RiDz9I2PpltrOnnIfDyg,bardo-lounge-and-supper-club-oakland,Bardo Lounge & Supper Club,https://s3-media0.fl.yelpcdn.com/bphoto/H03Q77...,https://www.yelp.com/biz/bardo-lounge-and-supp...,+15108368737,(510) 836-8737,$$$
EEQ_AjO0ErwoF1vkjm5-tg,EEQ_AjO0ErwoF1vkjm5-tg,lucuma-kitchen-and-bar-oakland,Lucuma Kitchen and Bar,https://s3-media0.fl.yelpcdn.com/bphoto/m8yVCu...,https://www.yelp.com/biz/lucuma-kitchen-and-ba...,+15106078110,(510) 607-8110,NaN
tljZfh7_dfmsU36o_vr_uw,tljZfh7_dfmsU36o_vr_uw,the-peach-oakland,The Peach,https://s3-media0.fl.yelpcdn.com/bphoto/RMrmJx...,https://www.yelp.com/biz/the-peach-oakland?adj...,+15109073892,(510) 907-3892,$$


# Activity 2: pull restaurants in a different location

- Try running a business search query for your hometown or another place by constructing a query similar to `yelp_genquery` but changing the location parameter
- Other endpoints require feeding what's called the business' fusion id into the API. Take an id from `yelp_stronly.id` and use the documentation here to pull the reviews for that business: https://docs.developer.yelp.com/reference/v3_business_reviews
- **Challenge**: generalize the previous step by writing a function that (1) takes a list of business ids as an input, (2) calls the reviews API for each id, (3) returns the results, and (4) rowbinds all results, i.e. turns them into a single, usable DataFrame

In [46]:
## use documentation to define what to search
## doc: https://www.yelp.com/developers/documentation/v3/business_search
## write the query 
base_url = "https://api.yelp.com/v3/businesses/search?"
my_name = "restaurants"
my_location = "Oakland,CA,94610"
yelp_genquery = ('{base_url}'
                'term={name}'
                '&location={loc}').format(base_url = base_url,
                name = my_name,
                loc = my_location)

## use requests to call the API; here, we're
## passing it our credentials (structure varies
## by API and telling it to only return 10 results
## (max is 50 at once)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_genresp = requests.get(yelp_genquery, headers = header)
yelp_genresp

## then, look at structure of response
yelp_genjson = yelp_genresp.json()

#yelp_genjson

business_id = yelp_genjson["businesses"][0]["id"]
print(business_id)
yelp_genquery=("https://api.yelp.com/v3/businesses/{business_id_or_alias}/reviews").format(business_id_or_alias = business_id)
header = {'Authorization': f"Bearer {API_KEY}"}
yelp_business = requests.get(yelp_genquery, headers = header)
yelp_business


<Response [200]>

-4fGYNfRa-hiiU1hCVFUEw


<Response [404]>

In [52]:
import requests
API_KEY = "rAZn1RX6Tr2o999Ly8EvD6ofsNGBZg3CMDsDxAwKGTEdWIi1zm9g23h_1ll5oix2Z0hSDiRloteTewU35nBqydGvoq0LngcGlZPP6LERVywiS_uc18KMmtvg9qQuZXYx"
url = "https://api.yelp.com/v3/businesses/Ecdu5qYM09F647Uoez99vg/reviews?"
header = {'Authorization': f'Bearer {API_KEY}'}
response = requests.get(url, headers=header)
print(response.text)

{"reviews": [{"id": "qUGeiKnzJaY4ldELiyUxVA", "url": "https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=qUGeiKnzJaY4ldELiyUxVA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A", "text": "The famous garlic knot pizza is a must try. My friend and I split a large (okay... way too big for 2 lol) and we both could barely finish 2 slices each...", "rating": 4.0, "time_created": "2026-06-26 11:48:37", "user": {"id": "kYf0AgTRNfhV6giPT3anpQ", "profile_url": "https://www.yelp.com/user_details?userid=kYf0AgTRNfhV6giPT3anpQ", "image_url": "https://s3-media0.fl.yelpcdn.com/photo/4-DvlQkWB00d-C7hfDpK2w/o.jpg", "name": "Cleo G."}}, {"id": "KMBiy0t5YTnmvLMR2i60rA", "url": "https://www.yelp.com/biz/ramuntos-brick-and-brew-pizzeria-hanover?adjust_creative=ZmitoFfF-pb6UbUwNTC3_A&hrid=KMBiy0t5YTnmvLMR2i60rA&utm_campaign=yelp_api_v3&utm_medium=api_v3_business_reviews&utm_source=ZmitoFfF-pb6UbUwNTC3_A",

In [53]:
def reviews(business_ids):
    all_reviews = []
    
    for business_id in business_ids:
        url = ("https://api.yelp.com/v3/businesses/{business_id_or_alias}/reviews").format(business_id_or_alias = business_id)
        response = requests.get(url, headers=header)
        review_list = response.json()["reviews"]
        
        df = pd.json_normalize(review_list)
        df["business_id"] = business_id
        all_reviews.append(df)
        
    return pd.concat(all_reviews, ignore_index=True)

In [57]:
business_ids = [b["id"] for b in yelp_genjson["businesses"]]

In [58]:
reviews(business_ids)

,id,url,text,rating,time_created,user.id,user.profile_url,user.image_url,user.name,business_id
0,Z3D_MIxHDyH6TWx0QkGdyw,https://www.yelp.com/biz/twinflame-oakland?adj...,Amazing loved the food 10/10!\nService is amaz...,5.0,2026-07-18 21:18:19,cofoGO_gazflr1z4HZe8VQ,https://www.yelp.com/user_details?userid=cofoG...,NaN,Nerida I.,-4fGYNfRa-hiiU1hCVFUEw
1,yujm46844ozbHrFLGjuTqA,https://www.yelp.com/biz/twinflame-oakland?adj...,"Twinflame on MacArthur, brand new spot just op...",5.0,2026-08-02 20:27:18,9lvYJ4OZZfF1Lwq7_V1pXA,https://www.yelp.com/user_details?userid=9lvYJ...,https://s3-media0.fl.yelpcdn.com/photo/G6lZpm4...,John S.,-4fGYNfRa-hiiU1hCVFUEw
2,WpXBcdY4v9yF6isHsffXnQ,https://www.yelp.com/biz/twinflame-oakland?adj...,Of course it was an instagram collab that got ...,3.0,2026-08-04 11:10:52,3roJp-RoqKVoMgdtuJFM9Q,https://www.yelp.com/user_details?userid=3roJp...,https://s3-media0.fl.yelpcdn.com/photo/I387jLz...,Christine M.,-4fGYNfRa-hiiU1hCVFUEw
3,S9mZ_ILg-eCYh0TZC2eGOQ,https://www.yelp.com/biz/almond-and-oak-oaklan...,I'm not sure if I was just extremely craving b...,5.0,2026-08-02 09:39:05,VXwiLvqT_qXp9T5iho81Tw,https://www.yelp.com/user_details?userid=VXwiL...,https://s3-media0.fl.yelpcdn.com/photo/5v3p2F8...,Kineshia C.,L7I_bhmMacR1lcVUuWwuKg
4,dS_H4tXtObCyow9yodz35w,https://www.yelp.com/biz/almond-and-oak-oaklan...,Came here for lunch with a friend at 1 PM on a...,4.0,2026-05-12 12:10:58,2PeqtdiehYm82TXymQUGow,https://www.yelp.com/user_details?userid=2Peqt...,https://s3-media0.fl.yelpcdn.com/photo/3OobBd7...,Michelle V.,L7I_bhmMacR1lcVUuWwuKg
5,GFTEQ4k_IAyEN0sPkWkZbQ,https://www.yelp.com/biz/almond-and-oak-oaklan...,Food was good.....service was good.\nHad break...,4.0,2026-04-28 10:25:00,VMHkA0mgHIl0N6YtGuC0hw,https://www.yelp.com/user_details?userid=VMHkA...,https://s3-media0.fl.yelpcdn.com/photo/jdoeChR...,Joy J.,L7I_bhmMacR1lcVUuWwuKg
6,N2GeEOvVnoqXQtkbqgzWzw,https://www.yelp.com/biz/mama-oakland-oakland?...,This is an excellent neighborhood place in a b...,5.0,2026-07-27 17:43:02,K7PYzne7jiH2Ufx_8zV8aA,https://www.yelp.com/user_details?userid=K7PYz...,https://s3-media0.fl.yelpcdn.com/photo/M0NmrLD...,Allure N.,D_yAQ4tHlWp2CJSqFCeW_w
7,IgM3qrAXrJCwkIoOHYgqNQ,https://www.yelp.com/biz/mama-oakland-oakland?...,"We've been coming here for years, and today, m...",4.0,2026-07-12 08:39:11,uHcoVFaNGNrPKVdjN5mvuQ,https://www.yelp.com/user_details?userid=uHcoV...,https://s3-media0.fl.yelpcdn.com/photo/lSRj6NX...,Ted R.,D_yAQ4tHlWp2CJSqFCeW_w
8,7DtOGck81mrzbE0gbHOy-A,https://www.yelp.com/biz/mama-oakland-oakland?...,Mama was recommended to me by a frugal foodie ...,5.0,2026-06-25 18:01:57,V_aMX7kxXf6GfNyFpO32rA,https://www.yelp.com/user_details?userid=V_aMX...,https://s3-media0.fl.yelpcdn.com/photo/KrR5dJA...,Catherine T.,D_yAQ4tHlWp2CJSqFCeW_w
9,xFRTScP-xdKrY5-TrPr0Lw,https://www.yelp.com/biz/sirene-oakland-2?adju...,Sirene absolutely knocked it out of the park.\...,5.0,2026-07-12 21:42:07,Rd873ATgu5aZ9wr2dkU6iQ,https://www.yelp.com/user_details?userid=Rd873...,https://s3-media0.fl.yelpcdn.com/photo/HRbbN9B...,Lin L.,5x-d9REDONrMabzr9GBm4A
